# GHOSTWIRE v1.1.0 — APK build on Google Colab
Compiles `ghostwire-1.1.0-arm64-v8a-debug.apk` from source: toolchain
install → SDK/NDK fetch → license accept → p4a patches → recipes →
Gradle → verified APK.

**Before you start:** Runtime → *Run all*. Keep the tab open; the
final build cell runs **30–60 minutes** (free CPU tier). No GPU needed.
When done, the APK downloads straight from the last cell.


## 1 · System packages (~2 min)


In [ ]:
!apt-get update -qq && apt-get install -qq -y git zip unzip autoconf libtool pkg-config zlib1g-dev libncurses-dev cmake libffi-dev libssl-dev python3-pip


## 2 · JDK 17 — Temurin (Colab's default Java is 11; Gradle needs 17)


In [ ]:
!cd /root && curl -sL -o jdk17.tar.gz "https://api.adoptium.net/v3/binary/latest/17/ga/linux/x64/jdk/hotspot/normal/eclipse" && tar xzf jdk17.tar.gz && /root/jdk-17*/bin/java -version


## 3 · Python build tools


In [ ]:
!pip3 install -q buildozer cython && buildozer --version && cython --version


## 4 · Get the source
Point this at your repo (or upload `ghostwire.zip` to `/content` and
use the commented-out lines instead).


In [ ]:
!cd /content && ( [ -d ghostwire/.git ] && git -C ghostwire pull || git clone https://github.com/YOU/ghostwire.git ) && ls ghostwire
# ---- zip alternative (upload ghostwire.zip first) ----
# !cd /content && rm -rf ghostwire && unzip -q ghostwire.zip && ls ghostwire


## 5 · Build environment (persists for every cell below)


In [ ]:
import os
os.environ["JAVA_HOME"] = "/root/jdk-17.0.20.1+1"
os.environ["PATH"] = "/root/jdk-17.0.20.1+1/bin:" + os.environ["PATH"]
os.environ["GRADLE_OPTS"] = "-Xmx1500m -XX:MaxMetaspaceSize=512m"
os.environ["KIVY_NO_ARGS"] = "1"
print("JAVA_HOME =", os.environ["JAVA_HOME"])


## 6 · Buildozer run #1 — fetch p4a + SDK/NDK (stops at licenses: expected)


In [ ]:
!cd /content/ghostwire && buildozer android debug > /tmp/b1.log 2>&1 || true
!tail -4 /tmp/b1.log
!grep -q "Aidl not found" /tmp/b1.log && echo "OK: reached license stage (expected stop)" || echo "UNEXPECTED: inspect /tmp/b1.log"


## 7 · Accept the Android SDK licenses (non-interactive)


In [ ]:
!SM=/root/.buildozer/android/platform/android-sdk/tools/bin/sdkmanager; yes | $SM --sdk_root=/root/.buildozer/android/platform/android-sdk --licenses 2>&1 | tail -2


## 8 · Apply the 4 GHOSTWIRE p4a patches


In [ ]:
!cd /content/ghostwire && python3 patches/apply_patches.py


## 9 · Buildozer run #2 — recipes + Gradle (**30–60 min**, milestones stream below)


In [ ]:
!cd /content/ghostwire && buildozer android debug 2>&1 | tee /tmp/b2.log | grep --line-buffered -E "Downloading|Recipe |-> running|BUILD|apk|available in the bin" | tail -60
!ls -la /content/ghostwire/bin/


## 10 · Verify + download the APK


In [ ]:
!cd /content/ghostwire && python3 patches/check_apk.py bin/ghostwire-1.1.0-arm64-v8a-debug.apk
from google.colab import files
files.download("/content/ghostwire/bin/ghostwire-1.1.0-arm64-v8a-debug.apk")
